# 스트레스 점수 예측 V22 — Verified 2D Pair Retuning

- Public 0.1272333 원본의 28개 2D Pair를 모두 유지
- 원본과 동일한 `float32` 순위·`np.argmin` 이웃 탐색
- Tree 분위수, Pair 분위수, Pair 비중, 반올림 여부만 탐색
- 3개 5-Fold 분할에서 모두 원본 V7보다 개선되고 평균 0.0005 이상 좋아야 선택
- Test 데이터 통계를 학습·선택에 사용하지 않음

In [1]:
from pathlib import Path
import os
from itertools import combinations

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 1. 설정

In [2]:
DATA_DIR=Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR=Path("/Users/bitsaem/Desktop/stress_project")
TARGET="stress_score";ID_COLUMN="ID"
N_ESTIMATORS=1200;MODEL_SEED=42;CV_SEEDS=(42,77,2026)
TREE_QUANTILES=(0.48,0.50,0.52,0.54,0.56)
PAIR_QUANTILES=(0.48,0.50,0.52,0.54,0.56)
PAIR_WEIGHTS=(0.05,0.10,0.15,0.20,0.25,0.30)
ROUND_OPTIONS=(False,True)
FEATURE_COPY_COUNTS={
 "mean_working":1,"bmi":4,"cholesterol":2,"height":2,
 "glucose":3,"weight":2,"cholesterol_glucose_ratio":2,"bone_density":4,
}
PAIR_COLUMNS=list(FEATURE_COPY_COUNTS)

## 2. 데이터와 파생변수

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test 입력 컬럼이 다릅니다.")
if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
    raise ValueError("ID 순서가 다릅니다.")

def add_features(frame):
    r=frame.copy(); hm=r["height"]/100
    sbp=r["systolic_blood_pressure"]; dbp=r["diastolic_blood_pressure"]
    r["bmi"]=r["weight"]/hm.pow(2)
    r["pulse_pressure"]=sbp-dbp
    r["mean_arterial_pressure"]=(sbp+2*dbp)/3
    r["blood_pressure_ratio"]=sbp/(dbp+1e-6)
    r["cholesterol_glucose_ratio"]=r["cholesterol"]/(r["glucose"]+1e-6)
    r["missing_count"]=r.isna().sum(axis=1)
    return r

def add_copies(frame):
    r=frame.copy()
    for f,n in FEATURE_COPY_COUNTS.items():
        for i in range(1,n+1): r[f"{f}__copy{i}"]=r[f]
    return r

y=train[TARGET].reset_index(drop=True)
X=add_features(train.drop(columns=[TARGET,ID_COLUMN])).reset_index(drop=True)
X_test=add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
print("Train:",X.shape,"| pairs:",len(list(combinations(PAIR_COLUMNS,2))))

Train: (3000, 22) | pairs: 28


## 3. ExtraTrees와 Train-기준 순위 함수

In [4]:
def make_preprocessor(frame):
    cat=frame.select_dtypes(include=["object","category","string"]).columns.tolist()
    num=frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("cat",Pipeline([
            ("imp",SimpleImputer(strategy="constant",fill_value="missing")),
            ("oh",OneHotEncoder(handle_unknown="ignore",sparse_output=False)),
        ]),cat),
        ("num",SimpleImputer(strategy="median",add_indicator=True),num),
    ])

def tree_material(train_x, valid_x, target):
    wt,wv=add_copies(train_x),add_copies(valid_x)
    p=make_preprocessor(wt); a=p.fit_transform(wt); b=p.transform(wv)
    m=ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,min_samples_leaf=1,max_features=1,
        random_state=MODEL_SEED,n_jobs=-1,
    )
    m.fit(a,target)
    return np.column_stack([t.predict(b) for t in m.estimators_])

def rank_transform(train_values, valid_values):
    median=np.nanmedian(train_values)
    a=np.nan_to_num(train_values,nan=median)
    b=np.nan_to_num(valid_values,nan=median)
    sorted_a=np.sort(a)
    train_rank=np.searchsorted(sorted_a,a,side="right")/len(a)
    valid_rank=np.searchsorted(sorted_a,b,side="right")/len(a)
    return train_rank.astype(np.float32),valid_rank.astype(np.float32)

## 4. 원본 float32 2D Pair 예측 소재

In [5]:
def pair_neighbor_material(train_x, valid_x, target):
    train_rank={}; valid_rank={}
    for col in PAIR_COLUMNS:
        train_rank[col],valid_rank[col]=rank_transform(
            train_x[col].to_numpy(float),valid_x[col].to_numpy(float)
        )
    target_values=target.to_numpy(float)
    materials=[]
    for first,second in combinations(PAIR_COLUMNS,2):
        distance=(
            np.abs(valid_rank[first][:,None]-train_rank[first][None,:])+
            np.abs(valid_rank[second][:,None]-train_rank[second][None,:])
        )
        nearest=np.argmin(distance,axis=1)
        materials.append(target_values[nearest])
    return np.column_stack(materials)

## 5. 3개 CV 분할의 Tree·Pair OOF 소재

In [6]:
split_data={}
for cv_seed in CV_SEEDS:
 cv=KFold(5,shuffle=True,random_state=cv_seed)
 tree_oof={q:np.zeros(len(y)) for q in TREE_QUANTILES}
 pair_oof={q:np.zeros(len(y)) for q in PAIR_QUANTILES}
 for fold,(ti,vi) in enumerate(cv.split(X),1):
    tm=tree_material(X.iloc[ti],X.iloc[vi],y.iloc[ti])
    pm=pair_neighbor_material(X.iloc[ti],X.iloc[vi],y.iloc[ti])
    for q in TREE_QUANTILES:
        tree_oof[q][vi]=np.clip(np.round(np.quantile(tm,q,axis=1),2),0,1)
    for q in PAIR_QUANTILES:
        pair_oof[q][vi]=np.quantile(pm,q,axis=1)
    print(f"CV {cv_seed} | Fold {fold}: complete")
 split_data[cv_seed]={"tree":tree_oof,"pair":pair_oof}
 baseline=np.clip(np.round(.85*tree_oof[.52]+.15*pair_oof[.52],2),0,1)
 print(f"CV {cv_seed} original V7 MAE: {mean_absolute_error(y,baseline):.6f}")

CV 42 | Fold 1: complete
CV 42 | Fold 2: complete
CV 42 | Fold 3: complete
CV 42 | Fold 4: complete
CV 42 | Fold 5: complete
CV 42 original V7 MAE: 0.148550
CV 77 | Fold 1: complete
CV 77 | Fold 2: complete
CV 77 | Fold 3: complete
CV 77 | Fold 4: complete
CV 77 | Fold 5: complete
CV 77 original V7 MAE: 0.146387
CV 2026 | Fold 1: complete
CV 2026 | Fold 2: complete
CV 2026 | Fold 3: complete
CV 2026 | Fold 4: complete
CV 2026 | Fold 5: complete
CV 2026 original V7 MAE: 0.144733


## 6. 원본 V7보다 확실히 좋은 조합만 선택

In [7]:
rows=[]
for tree_q in TREE_QUANTILES:
 for pair_q in PAIR_QUANTILES:
  for pair_weight in PAIR_WEIGHTS:
   for rounding in ROUND_OPTIONS:
    scores=[];improvements=[]
    for cv_seed in CV_SEEDS:
        item=split_data[cv_seed]
        baseline=np.clip(np.round(.85*item["tree"][.52]+.15*item["pair"][.52],2),0,1)
        pred=(1-pair_weight)*item["tree"][tree_q]+pair_weight*item["pair"][pair_q]
        if rounding:pred=np.round(pred,2)
        pred=np.clip(pred,0,1)
        b=mean_absolute_error(y,baseline);s=mean_absolute_error(y,pred)
        scores.append(s);improvements.append(b-s)
    rows.append({"tree_quantile":tree_q,"pair_quantile":pair_q,
      "pair_weight":pair_weight,"rounding":rounding,
      "mean_CV_MAE":np.mean(scores),"mean_improvement":np.mean(improvements),
      "worst_improvement":min(improvements),"wins":sum(v>0 for v in improvements)})
results=pd.DataFrame(rows).sort_values(
 ["wins","worst_improvement","mean_CV_MAE"],ascending=[False,False,True]
).reset_index(drop=True)
eligible=results[(results.wins==3)&(results.worst_improvement>0)&(results.mean_improvement>=.0005)]
if len(eligible):selected=eligible.iloc[0]
else:selected=results[(results.tree_quantile==.52)&(results.pair_quantile==.52)&
 (results.pair_weight==.15)&(results.rounding)].iloc[0]
print("Selected tree quantile:",selected.tree_quantile)
print("Selected pair quantile / weight:",selected.pair_quantile,"/",selected.pair_weight)
print("Round to 2 decimals:",bool(selected.rounding))
print("Repeated-CV mean MAE:",f"{selected.mean_CV_MAE:.6f}")
print("Mean improvement:",f"{selected.mean_improvement:+.6f}")
print("Worst split improvement:",f"{selected.worst_improvement:+.6f}")
print("Wins:",int(selected.wins),"/ 3")
results.head(20)

Selected tree quantile: 0.52
Selected pair quantile / weight: 0.52 / 0.15
Round to 2 decimals: True
Repeated-CV mean MAE: 0.146557
Mean improvement: +0.000000
Worst split improvement: +0.000000
Wins: 0 / 3


,tree_quantile,pair_quantile,pair_weight,rounding,mean_CV_MAE,mean_improvement,worst_improvement,wins
0,0.50,0.56,0.30,True,0.146409,0.000148,0.000080,3
1,0.50,0.56,0.25,True,0.146423,0.000133,0.000047,3
2,0.50,0.56,0.30,False,0.146433,0.000123,0.000040,3
3,0.52,0.50,0.15,True,0.146491,0.000066,0.000037,3
4,0.52,0.52,0.20,True,0.146494,0.000062,0.000033,3
5,0.50,0.56,0.25,False,0.146429,0.000128,0.000025,3
6,0.50,0.54,0.30,True,0.146471,0.000086,0.000013,3
7,0.52,0.52,0.25,True,0.146468,0.000089,0.000003,3
8,0.52,0.50,0.20,True,0.146421,0.000136,0.000000,2
9,0.50,0.54,0.30,False,0.146484,0.000073,-0.000003,2


## 7. 전체 Train으로 Test 예측

In [8]:
tm=tree_material(X,X_test,y)
pm=pair_neighbor_material(X,X_test,y)
pred_tree=np.clip(np.round(np.quantile(tm,float(selected.tree_quantile),axis=1),2),0,1)
pred_pair=np.quantile(pm,float(selected.pair_quantile),axis=1)
pred_v22=(1-float(selected.pair_weight))*pred_tree+float(selected.pair_weight)*pred_pair
if bool(selected.rounding):pred_v22=np.round(pred_v22,2)
pred_v22=np.clip(pred_v22,0,1)
print("Prediction range:",pred_v22.min(),pred_v22.max())
print("첫 10개:",pred_v22[:10])

Prediction range: 0.0 1.0
첫 10개: [0.56 0.97 0.19 0.48 0.53 0.47 0.78 0.48 0.57 0.88]


## 8. 제출 파일

In [9]:
submission_v22=submission.copy();submission_v22[TARGET]=pred_v22
output_path=OUTPUT_DIR/"submit_v22_verified_pair_retuning.csv"
submission_v22.to_csv(output_path,index=False,encoding="utf-8")
print("Saved:",output_path)
submission_v22.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_v22_verified_pair_retuning.csv


,ID,stress_score
0,TEST_0000,0.56
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.48
4,TEST_0004,0.53
5,TEST_0005,0.47
6,TEST_0006,0.78
7,TEST_0007,0.48
8,TEST_0008,0.57
9,TEST_0009,0.88


## 제출 기준

- `Wins: 3 / 3`
- `Mean improvement >= 0.0005`
- `Worst split improvement > 0`
- 세 조건을 모두 만족할 때만 V22 제출 후보
- 실패하면 Public 0.1272333 원본 V7 유지